# Phase 1: Environment Setup & Data Loading

## Overview
In this initial step, we set up the data science environment by importing the required libraries for data analysis, visualization, and preprocessing. We then load the raw dataset to inspect its structure and dimensions before starting the preprocessing pipeline.

### Steps Performed:
1. **Import Libraries:** `pandas`, `numpy`, `matplotlib`, `seaborn`, and `scikit-learn` modules.
2. **Load Dataset:** Load the `heart_disease_health_indicators_BRFSS2015.csv` file from the raw data directory.
3. **Initial Inspection:** Verify the shape (rows and columns) and preview the top rows of the dataset.


# Phase 2: Train-Test Split (Preventing Data Leakage)

## Overview
Before applying any data imputation, transformation, or cleaning, we split the dataset into Training (80%) and Testing (20%) sets. 

### Why Split First?
Fitting imputation or scaling parameters on the full dataset before splitting leads to **data leakage**, resulting in overly optimistic model evaluations. By splitting early, all preprocessing rules will be learned strictly from the training set.

### Steps Performed:
1. Separate features (`X`) and the target variable (`y = HeartDiseaseorAttack`).
2. Perform an 80/20 train-test split with stratification to maintain target class proportions.

In [16]:
# Separate target variable from features
X = df.drop(columns=['HeartDiseaseorAttack'])
y = df['HeartDiseaseorAttack']

# Train-Test Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training Features Shape: {X_train.shape}")
print(f"Testing Features Shape:  {X_test.shape}")
print(f"Training Target Class Distribution:\n{y_train.value_counts(normalize=True)}")

NameError: name 'train_test_split' is not defined

# Phase 3: Data Audit & Cleaning

## Overview
In this step, we perform a thorough data quality audit on the training set (`X_train`) to identify missing values, check summary statistics, and detect duplicate rows or invalid data ranges before proceeding to EDA.

### Steps Performed:
1. Check for missing/null values across all features.
2. Evaluate summary statistics (min, max, mean, standard deviation) for potential anomalies or outliers.
3. Check and remove any duplicate entries from the training set.

In [ ]:
# 1. Missing values check
missing_vals = X_train.isnull().sum()
print("Missing values per column in X_train:")
print(missing_vals[missing_vals > 0] if missing_vals.sum() > 0 else "No missing values found!")
# 2. Check for duplicate rows in training set
duplicates_count = X_train.duplicated().sum()
print(f"\nDuplicate rows in X_train: {duplicates_count}")
# 3. Summary statistics for range/outlier checks
print("\nSummary Statistics of X_train:")
X_train.describe().T[['mean', 'std', 'min', '50%', 'max']]

# Phase 4: Duplicate Removal & Exploratory Data Analysis (EDA)

## Overview
We remove duplicate rows from the training dataset (`X_train` and `y_train`) to ensure model training relies on unique instances. Following the cleanup, we explore feature distributions, class imbalances in the target variable, and correlations among health indicators.

### Steps Performed:
1. Deduplicate training data synchronized with target labels.
2. Plot target variable distribution (`HeartDiseaseorAttack`).
3. Visualize continuous features (e.g., `BMI`) distribution.
4. Generate a Correlation Heatmap to observe relationships between features.

In [ ]:
# 1. Deduplicate X_train and synchronize y_train
train_combined = pd.concat([X_train, y_train], axis=1)
train_combined = train_combined.drop_duplicates()

# Re-split X_train and y_train after deduplication
X_train = train_combined.drop(columns=['HeartDiseaseorAttack'])
y_train = train_combined['HeartDiseaseorAttack']

print(f"Shape of X_train after deduplication: {X_train.shape}")

# 2. EDA Visualizations Setup
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Target Variable Distribution
sns.countplot(x=y_train, ax=axes[0], palette='coolwarm')
axes[0].set_title('Target Distribution (HeartDiseaseorAttack)')
axes[0].set_xlabel('Heart Disease (0 = No, 1 = Yes)')
axes[0].set_ylabel('Count')

# BMI Distribution
sns.histplot(X_train['BMI'], kde=True, bins=30, ax=axes[1], color='teal')
axes[1].set_title('BMI Feature Distribution')
axes[1].set_xlabel('BMI')

plt.tight_layout()
plt.show()

# Correlation Heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(X_train.corr(), cmap='Blues', annot=False, cbar=True)
plt.title('Feature Correlation Matrix')
plt.show()

# Phase 5: Exporting Cleaned & Processed Datasets

## Overview
In this final step, we export the split and cleaned datasets to the `Data/processed/` directory. This ensures that down-stream modeling (e.g., Student 3's task) can directly consume preprocessed data without risk of data leakage.

### Saved Files:
* `X_train_clean.csv` & `y_train_clean.csv`
* `X_test_clean.csv` & `y_test_clean.csv`

In [ ]:
import os
from sklearn.model_selection import train_test_split

X = df.drop(columns=['HeartDiseaseorAttack'])
y = df['HeartDiseaseorAttack']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
# Processed directory path create karein agar exist nahi karti
processed_dir = '../Data/processed'
os.makedirs(processed_dir, exist_ok=True)

# Export Dataset
X_train.to_csv(f'{processed_dir}/X_train_clean.csv', index=False)
y_train.to_csv(f'{processed_dir}/y_train_clean.csv', index=False)
X_test.to_csv(f'{processed_dir}/X_test_clean.csv', index=False)
y_test.to_csv(f'{processed_dir}/y_test_clean.csv', index=False)

print("✅ All clean datasets successfully exported to 'Data/processed/' directory!")

# FEATURE ENGINEERING

In [ ]:
import pandas as pd
import numpy as np
df = pd.read_csv("C:/Users/srhfu/Downloads/Programming/heart_disease_health_indicators_BRFSS2015.csv")

print("✅ Dataset loaded successfully!")
print(df.shape)
df.head()

# 1. Clinical Composite: Cardiometabolic Risk Score
Aggregates major CDC-recognized cardiometabolic risk factors into a single interpretable score.
Higher values indicate elevated chronic disease burden

In [ ]:
df['CardiometabolicRiskScore'] = (
    df['HighBP'] +
    df['HighChol'] +
    df['Diabetes'] +
    df['Stroke'] +
    df['HeartDiseaseorAttack']
)

# 2. Obesity Classification (BMI-Based)
 Converts continuous BMI into a binary clinical obesity flag.

In [ ]:
df['ObesityFlag'] = (df['BMI'] >= 30).astype(int)

# 3. Tobacco Exposure Index
Combines smoking status and frequency into a single exposure metric.

In [ ]:
# 1 = current smoker, 2 = former smoker
df['CurrentSmokerFlag'] = (df['Smoker'] == 1).astype(int)
df['FormerSmokerFlag'] = (df['Smoker'] == 2).astype(int)


 # 4. Alcohol Misuse Indicator
Heavy drinking is defined by CDC thresholds. Converts to binary flag.

In [ ]:
df['HeavyDrinkFlag'] = (df['HvyAlcoholConsump'] == 1).astype(int)

# 5. Mental Health Burden Score
Aggregates poor mental health days and depression diagnosis.

In [17]:
df.columns
df['MentalHealthBurden'] = (
    df.get('MentHlth', 0) +
    df.get('Depressed', 0)
)


# 6. Physical Activity Adequacy
Converts activity level into a binary adequacy flag.

In [18]:
df['PhysicallyActiveFlag'] = (df['PhysActivity'] == 1).astype(int)

# 7. Respiratory Risk Composite
Combines asthma and COPD into a single respiratory risk indicator.

In [25]:

df['RespiratoryRiskScore'] = (
    df.get('Asthma', 0) +
    df.get('COPD', 0)
)


# 8. Hypertension Severity Proxy
Combines high BP with medication status (if available).

In [26]:
if 'BP_Medication' in df.columns:
    df['HypertensionSeverity'] = (
        df['HighBP'] +
        df['BP_Medication']
    )

# 9.Metabolic Syndrome Proxy
Uses obesity, high BP, high cholesterol, and diabetes.

In [27]:
df['MetabolicSyndromeProxy'] = (
    df['ObesityFlag'] +
    df['HighBP'] +
    df['HighChol'] +
    df['Diabetes']
)

# 10. Age Risk Tiering
Converts continuous age into clinically meaningful risk tiers.

In [28]:
df['AgeRiskTier'] = pd.cut(
    df['Age'],
    bins=[0, 34, 49, 64, 120],
    labels=['Low', 'Moderate', 'High', 'Very High']
)

# 11. Healthcare Access Risk
Captures insurance status + inability to see doctor due to cost.

In [33]:
df['HealthcareAccessRisk'] = (
    (df.get('NoDocbcCost', 0) == 1).astype(int)
    if isinstance(df.get('NoDocbcCost', None), pd.Series) else int(df.get('NoDocbcCost', 0) == 1)
) + (
    (df.get('NoHealthInsurance', 0) == 1).astype(int)
    if isinstance(df.get('NoHealthInsurance', None), pd.Series) else int(df.get('NoHealthInsurance', 0) == 1)
)


# 12. Cholesterol Treatment Gap
Identifies individuals with high cholesterol but no medication.

In [34]:
if 'Chol_Medication' in df.columns:
    df['CholTreatmentGap'] = (
        (df['HighChol'] == 1).astype(int) -
        (df['Chol_Medication'] == 1).astype(int)
    )
else:
    df['CholTreatmentGap'] = df['HighChol']

# 13. Social Isolation Proxy
Uses self-reported general health + mental health days.

In [35]:
df['SocialIsolationProxy'] = (
    (df['GenHlth'] >= 4).astype(int) +
    (df['MentHlth'] > 10).astype(int)
)

# 14. Final Check: Replace Missing Values
Ensures all engineered features are clean for modeling.

In [37]:
df.fillna(0, inplace=True)

print("✅ Feature engineering completed successfully!")

✅ Feature engineering completed successfully!


In [39]:
df.head(20)

,HeartDiseaseorAttack,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,Diabetes,PhysActivity,Fruits,...,FormerSmokerFlag,HeavyDrinkFlag,MentalHealthBurden,PhysicallyActiveFlag,RespiratoryRiskScore,MetabolicSyndromeProxy,AgeRiskTier,HealthcareAccessRisk,CholTreatmentGap,SocialIsolationProxy
0,0.0,1.0,1.0,1.0,40.0,1.0,0.0,0.0,0.0,0.0,...,0,0,18.0,0,0,3.0,Low,0,1.0,2
1,0.0,0.0,0.0,0.0,25.0,1.0,0.0,0.0,1.0,0.0,...,0,0,0.0,1,0,0.0,Low,1,0.0,0
2,0.0,1.0,1.0,1.0,28.0,0.0,0.0,0.0,0.0,1.0,...,0,0,30.0,0,0,2.0,Low,1,1.0,2
3,0.0,1.0,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,...,0,0,0.0,1,0,1.0,Low,0,0.0,0
4,0.0,1.0,1.0,1.0,24.0,0.0,0.0,0.0,1.0,1.0,...,0,0,3.0,1,0,2.0,Low,0,1.0,0
5,0.0,1.0,1.0,1.0,25.0,1.0,0.0,0.0,1.0,1.0,...,0,0,0.0,1,0,2.0,Low,0,1.0,0
6,0.0,1.0,0.0,1.0,30.0,1.0,0.0,0.0,0.0,0.0,...,0,0,0.0,0,0,2.0,Low,0,0.0,0
7,0.0,1.0,1.0,1.0,25.0,1.0,0.0,0.0,1.0,0.0,...,0,0,0.0,1,0,2.0,Low,0,1.0,0
8,1.0,1.0,1.0,1.0,30.0,1.0,0.0,2.0,0.0,1.0,...,0,0,30.0,0,0,5.0,Low,0,1.0,2
9,0.0,0.0,0.0,1.0,24.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0.0,0,0,0.0,Low,0,0.0,0


In [42]:
df.describe(include='all')


,HeartDiseaseorAttack,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,Diabetes,PhysActivity,Fruits,...,FormerSmokerFlag,HeavyDrinkFlag,MentalHealthBurden,PhysicallyActiveFlag,RespiratoryRiskScore,MetabolicSyndromeProxy,AgeRiskTier,HealthcareAccessRisk,CholTreatmentGap,SocialIsolationProxy
count,253680.000000,253680.000000,253680.000000,253680.000000,253680.000000,253680.000000,253680.000000,253680.000000,253680.000000,253680.000000,...,253680.0,253680.000000,253680.000000,253680.000000,253680.0,253680.00000,253680,253680.000000,253680.000000,253680.000000
unique,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,1,NaN,NaN,NaN
top,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,Low,NaN,NaN,NaN
freq,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,253680,NaN,NaN,NaN
mean,0.094186,0.429001,0.424121,0.962670,28.382364,0.443169,0.040571,0.296921,0.756544,0.634256,...,0.0,0.056197,3.184772,0.756544,0.0,1.49635,NaN,0.084177,0.424121,0.270687
std,0.292087,0.494934,0.494210,0.189571,6.608694,0.496761,0.197294,0.698160,0.429169,0.481639,...,0.0,0.230302,7.412847,0.429169,0.0,1.39940,NaN,0.277654,0.494210,0.541551
min,0.000000,0.000000,0.000000,0.000000,12.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.000000,0.000000,0.000000,0.0,0.00000,NaN,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,1.000000,24.000000,0.000000,0.000000,0.000000,1.000000,0.000000,...,0.0,0.000000,0.000000,1.000000,0.0,0.00000,NaN,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,1.000000,27.000000,0.000000,0.000000,0.000000,1.000000,1.000000,...,0.0,0.000000,0.000000,1.000000,0.0,1.00000,NaN,0.000000,0.000000,0.000000
75%,0.000000,1.000000,1.000000,1.000000,31.000000,1.000000,0.000000,0.000000,1.000000,1.000000,...,0.0,0.000000,2.000000,1.000000,0.0,2.00000,NaN,0.000000,1.000000,0.000000


In [47]:
df.shape
df.columns
df.isna().sum()


HeartDiseaseorAttack        0
HighBP                      0
HighChol                    0
CholCheck                   0
BMI                         0
Smoker                      0
Stroke                      0
Diabetes                    0
PhysActivity                0
Fruits                      0
Veggies                     0
HvyAlcoholConsump           0
AnyHealthcare               0
NoDocbcCost                 0
GenHlth                     0
MentHlth                    0
PhysHlth                    0
DiffWalk                    0
Sex                         0
Age                         0
Education                   0
Income                      0
CardiometabolicRiskScore    0
ObesityFlag                 0
CurrentSmokerFlag           0
FormerSmokerFlag            0
HeavyDrinkFlag              0
MentalHealthBurden          0
PhysicallyActiveFlag        0
RespiratoryRiskScore        0
MetabolicSyndromeProxy      0
AgeRiskTier                 0
HealthcareAccessRisk        0
CholTreatm

In [48]:
pd.DataFrame({
    'CardiometabolicRiskScore': df.get('CardiometabolicRiskScore'),
    'ObesityFlag': df.get('ObesityFlag'),
    'TobaccoExposureIndex': df.get('TobaccoExposureIndex'),
    'MentalHealthBurden': df.get('MentalHealthBurden'),
    'RespiratoryRiskScore': df.get('RespiratoryRiskScore'),
    'MetabolicSyndromeProxy': df.get('MetabolicSyndromeProxy'),
    'HealthcareAccessRisk': df.get('HealthcareAccessRisk')
}).describe()


,CardiometabolicRiskScore,ObesityFlag,MentalHealthBurden,RespiratoryRiskScore,MetabolicSyndromeProxy,HealthcareAccessRisk
count,253680.000000,253680.000000,253680.000000,253680.0,253680.00000,253680.000000
mean,1.284800,0.346306,3.184772,0.0,1.49635,0.084177
std,1.364282,0.475793,7.412847,0.0,1.39940,0.277654
min,0.000000,0.000000,0.000000,0.0,0.00000,0.000000
25%,0.000000,0.000000,0.000000,0.0,0.00000,0.000000
50%,1.000000,0.000000,0.000000,0.0,1.00000,0.000000
75%,2.000000,1.000000,2.000000,0.0,2.00000,0.000000
max,6.000000,1.000000,30.000000,0.0,5.00000,1.000000


In [49]:
[col for col in df.columns if col not in [
    'HighBP','HighChol','BMI','Smoker','MentHlth','Depressed','Asthma','COPD'
]]


['HeartDiseaseorAttack',
 'CholCheck',
 'Stroke',
 'Diabetes',
 'PhysActivity',
 'Fruits',
 'Veggies',
 'HvyAlcoholConsump',
 'AnyHealthcare',
 'NoDocbcCost',
 'GenHlth',
 'PhysHlth',
 'DiffWalk',
 'Sex',
 'Age',
 'Education',
 'Income',
 'CardiometabolicRiskScore',
 'ObesityFlag',
 'CurrentSmokerFlag',
 'FormerSmokerFlag',
 'HeavyDrinkFlag',
 'MentalHealthBurden',
 'PhysicallyActiveFlag',
 'RespiratoryRiskScore',
 'MetabolicSyndromeProxy',
 'AgeRiskTier',
 'HealthcareAccessRisk',
 'CholTreatmentGap',
 'SocialIsolationProxy']